اگربات ابھی تک نہیں کی ہے تو درج ذیل نوٹ بکس چلانے کے لیے، آپ کو ایک ایسا ماڈل تعینات کرنا ہوگا جو `text-embedding-ada-002` کو بنیادی ماڈل کے طور پر استعمال کرتا ہو اور .env فائل کے اندر تعیناتی کا نام `AZURE_OPENAI_EMBEDDINGS_ENDPOINT` کے طور پر سیٹ کریں۔


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


اگلے مرحلے میں، ہم Embedding Index کو Pandas Dataframe میں لوڈ کرنے جا رہے ہیں۔ Embedding Index ایک JSON فائل میں ذخیرہ کیا گیا ہے جس کا نام `embedding_index_3m.json` ہے۔ Embedding Index میں YouTube ٹرانسکرپٹس کے لیے ایمبیڈنگز شامل ہیں جو اکتوبر 2023 کے آخر تک کے ہیں۔


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

اگلے مرحلے میں، ہم `get_videos` نام کا ایک فنکشن بنائیں گے جو کہ کوری کے لیے ایمبیڈنگ انڈیکس میں تلاش کرے گا۔ یہ فنکشن وہ ٹاپ 5 ویڈیوز واپس کرے گا جو کوری کے سب سے زیادہ مماثل ہوں۔ فنکشن درج ذیل طریقے سے کام کرتا ہے:

1. سب سے پہلے، ایمبیڈنگ انڈیکس کی ایک کاپی بنائی جاتی ہے۔
2. اس کے بعد، اوپن اے آئی ایمبیڈنگ API کی مدد سے کوری کے لیے ایمبیڈنگ کا حساب لگایا جاتا ہے۔
3. پھر ایمبیڈنگ انڈیکس میں ایک نیا کالم `similarity` بنایا جاتا ہے۔ `similarity` کالم میں کوری ایمبیڈنگ اور ہر ویڈیو حصے کی ایمبیڈنگ کے درمیان کاسائن مشابہت ہوتی ہے۔
4. اگلا قدم یہ ہے کہ ایمبیڈنگ انڈیکس کو `similarity` کالم کی بنیاد پر فلٹر کیا جاتا ہے۔ ایمبیڈنگ انڈیکس کو صرف ایسی ویڈیوز پر محدود کیا جاتا ہے جن کی کاسائن مشابہت 0.75 یا اس سے زیادہ ہو۔
5. آخر میں، ایمبیڈنگ انڈیکس کو `similarity` کالم کی بنیاد پر ترتیب دیا جاتا ہے اور ٹاپ 5 ویڈیوز واپس کی جاتی ہیں۔


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

یہ فعل بہت سادہ ہے، یہ صرف تلاش کے سوال کے نتائج کو پرنٹ کرتا ہے۔


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. سب سے پہلے، ایمبیڈنگ انڈیکس کو پانڈا ڈیٹافریم میں لوڈ کیا جاتا ہے۔
2. اس کے بعد، صارف کو ایک استفسار داخل کرنے کے لیے کہا جاتا ہے۔
3. پھر `get_videos` فنکشن کو کال کیا جاتا ہے تاکہ استفسار کے لیے ایمبیڈنگ انڈیکس میں تلاش کی جا سکے۔
4. آخر میں، `display_results` فنکشن کو کال کیا جاتا ہے تاکہ نتائج صارف کو دکھائے جا سکیں۔
5. اس کے بعد صارف کو ایک اور استفسار داخل کرنے کے لیے کہا جاتا ہے۔ یہ عمل جاری رہتا ہے جب تک کہ صارف `exit` نہ لکھے۔

![](../../../../translated_images/ur/notebook-search.1e320b9c7fcbb0bc.webp)

آپ سے استفسار داخل کرنے کو کہا جائے گا۔ ایک استفسار درج کریں اور انٹر دبائیں۔ ایپلیکیشن آپ کو استفسار سے متعلق ویڈیوز کی فہرست واپس کرے گی۔ ایپلیکیشن ویڈیو میں اس جگہ کا لنک بھی واپس کرے گی جہاں سوال کا جواب موجود ہے۔

یہاں کچھ استفسارات ہیں جو آپ آزما سکتے ہیں:

- Azure Machine Learning کیا ہے؟
- Convolutional neural networks کیسے کام کرتے ہیں؟
- Neural network کیا ہے؟
- کیا میں Azure Machine Learning کے ساتھ Jupyter Notebooks استعمال کر سکتا ہوں؟
- ONNX کیا ہے؟


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
